# Bài 14: LLM-based Router
## Thay keyword matching bằng phân tích ngữ nghĩa

**Mục tiêu:**
- Hiểu tại sao keyword router brittle
- Dùng LLM trả về JSON để quyết định agent nào cần chạy
- Tích hợp LLM router vào LangGraph orchestrator

---
## Phần 1: Lý thuyết

### Vấn đề của Keyword Router (Bài 11)

```python
def router_node(state):
    question = state['question'].lower()
    if "tin tức" in question or "news" in question:
        agents.append("web_research")
    if "báo cáo" in question or "revenue" in question:
        agents.append("rag")
    ...
```

**3 điểm yếu:**

1. **Phụ thuộc từ khoá cụ thể** — `"What happened to NVDA last week?"` không chứa `"news"` nhưng rõ ràng cần `web_research`
2. **Brittle với paraphrase** — `"quarterly earnings"` ≠ `"revenue"` ≠ `"doanh thu"` dù nghĩa như nhau
3. **Fallback "chạy cả 3"** — khi không khớp keyword nào, router chạy 3 agent → tốn tiền, chậm

---

### Giải pháp: LLM Router

Ta hỏi LLM: *"Câu hỏi này cần agent nào?"* và yêu cầu nó trả về JSON.

```
Input:  "What happened to NVDA last week?"
Output: {"agents": ["web_research"]}

Input:  "Giá NVDA hôm nay và doanh thu Q1?"
Output: {"agents": ["financial_data", "rag"]}
```

LLM hiểu **ngữ nghĩa** thay vì đếm từ. Không cần liệt kê từng biến thể ngôn ngữ.

### Demo: Keyword Router thất bại ở đâu

Chạy cell dưới để thấy các câu hỏi mà keyword router xử lý sai:

In [1]:
# Keyword router từ Bài 11 — KHÔNG sửa gì cả
def keyword_router(question: str) -> list[str]:
    q = question.lower()
    agents = []
    if "tin tức" in q or "news" in q:
        agents.append("web_research")
    if "giá" in q or "price" in q:
        agents.append("financial_data")
    if "báo cáo" in q or "doanh thu" in q or "report" in q or "revenue" in q:
        agents.append("rag")
    if not agents:
        agents = ["web_research", "financial_data", "rag"]  # fallback tốn kém
    return agents


# Các câu hỏi test — chú ý câu nào bị route sai
test_questions = [
    "What happened to NVDA last week?",           # cần web_research
    "NVDA quarterly earnings là bao nhiêu?",      # cần rag
    "How is NVIDIA stock performing recently?",   # cần web_research + financial_data
    "Cho tôi biết mọi thứ về NVDA",              # ambiguous, nên chạy cả 3
    "NVDA stock went up or down?",                # cần financial_data + web_research
]

print("=" * 60)
print("KEYWORD ROUTER KẾT QUẢ:")
print("=" * 60)
for q in test_questions:
    result = keyword_router(q)
    print(f"Q: {q}")
    print(f"→ {result}")
    print()

KEYWORD ROUTER KẾT QUẢ:
Q: What happened to NVDA last week?
→ ['web_research', 'financial_data', 'rag']

Q: NVDA quarterly earnings là bao nhiêu?
→ ['web_research', 'financial_data', 'rag']

Q: How is NVIDIA stock performing recently?
→ ['web_research', 'financial_data', 'rag']

Q: Cho tôi biết mọi thứ về NVDA
→ ['web_research', 'financial_data', 'rag']

Q: NVDA stock went up or down?
→ ['web_research', 'financial_data', 'rag']



**Nhận xét:** Hầu hết các câu trên đều bị route về fallback `["web_research", "financial_data", "rag"]` — chạy cả 3 dù không cần thiết.

---
## Phần 2: Ví dụ — LLM Router độc lập

Trước khi tích hợp vào LangGraph, ta viết một hàm `llm_router()` độc lập và test nó.

**Kỹ thuật quan trọng: JSON output từ LLM**

Gemini có thể bọc JSON trong markdown code fence:
````
```json
{"agents": ["web_research"]}
```
````
→ Ta cần strip fence trước khi `json.loads()`.

Nếu LLM trả về JSON không hợp lệ → fallback về `["web_research", "financial_data", "rag"]`.

In [2]:
import os, json
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [12]:
# VÍ DỤ: hàm llm_router() độc lập — đọc kỹ trước khi làm bài tập

ROUTER_PROMPT = """\
Bạn là routing assistant cho hệ thống phân tích cổ phiếu.
Dựa vào câu hỏi, quyết định cần chạy agent nào.

Các agent có sẵn:
- web_research  : tin tức, sự kiện gần đây, tâm lý thị trường, diễn biến mới nhất
- financial_data: giá cổ phiếu, P/E, volume, market cap, chỉ số tài chính hiện tại
- rag           : báo cáo tài chính, doanh thu, lợi nhuận theo quý/năm trong file PDF

Quy tắc:
1. Chọn ÍT NHẤT 1 agent
2. Chỉ chọn những agent THỰC SỰ cần thiết
3. Trả về JSON DUY NHẤT, không thêm giải thích

Format: {{"agents": ["tên_agent_1", "tên_agent_2"]}}

Câu hỏi: {question}
"""

VALID_AGENTS = {"web_research", "financial_data", "rag"}


def llm_router(question: str) -> list[str]:
    """Dùng LLM phân tích intent và trả về list agent cần chạy."""
    prompt = ROUTER_PROMPT.format(question=question)

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",   # đổi từ flash-lite
            contents=prompt
        )

        text = response.text.strip()

        # Strip markdown code fence nếu có
        if text.startswith("```"):
            lines = text.split("\n")
            text = "\n".join(lines[1:-1])  # bỏ dòng đầu (```) và dòng cuối (```)

        data = json.loads(text)
        agents = data.get("agents", [])

        # Lọc tên agent hợp lệ
        agents = [a for a in agents if a in VALID_AGENTS]

        if not agents:
            raise ValueError("LLM trả về list agent rỗng sau khi lọc")

        return agents

    except Exception as e:
        print(f"[Router Error] {e} → fallback: chạy cả 3")
        return ["web_research", "financial_data", "rag"]

In [18]:
# Test llm_router với cùng câu hỏi ở trên
print("=" * 60)
print("LLM ROUTER KẾT QUẢ:")
print("=" * 60)
for q in test_questions:
    result = llm_router(q)
    print(f"Q: {q}")
    print(f"→ {result}")
    print()

# So sánh với keyword router ở trên — LLM có chọn đúng hơn không?

LLM ROUTER KẾT QUẢ:
Q: What happened to NVDA last week?
→ ['web_research', 'financial_data']

Q: NVDA quarterly earnings là bao nhiêu?
→ ['rag']

Q: How is NVIDIA stock performing recently?
→ ['financial_data', 'web_research']

Q: Cho tôi biết mọi thứ về NVDA
→ ['web_research', 'financial_data', 'rag']

Q: NVDA stock went up or down?
→ ['financial_data']



---
## Phần 3: Bài tập — Tích hợp LLM Router vào LangGraph

Graph structure giữ nguyên như Bài 11:
```
START → router → [web_research | financial_data | rag] → analysis → END
```

Việc cần làm:
- ✅ `OrchestratorState` và agent stubs — đã cho sẵn
- ✅ `route_to_agents()`, các node wrapper — đã cho sẵn  
- **❓ Bước 1:** Viết `llm_router_node(state)` — gọi `llm_router()`, cập nhật state
- **❓ Bước 2:** Build LangGraph với LLM router
- **❓ Bước 3:** Test với các câu hỏi mà keyword router đã fail

In [13]:
# ĐÃ CHO SẴN — State + agent stubs
import yfinance as yf
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class OrchestratorState(TypedDict):
    question: str
    symbol: str
    needed_agents: list[str]
    web_result: str
    financial_result: str
    rag_result: str
    final_result: str


# Agent stubs — trả về placeholder, thay bằng agent thật nếu muốn
def web_research_agent(question: str) -> str:
    return f"[Web stub] Tìm kiếm: '{question[:60]}' — thay bằng DuckDuckGo search thật"


def financial_data_agent(symbol: str) -> str:
    ticker = yf.Ticker(symbol)
    price = ticker.info.get("currentPrice", "N/A")
    market_cap = ticker.info.get("marketCap", "N/A")
    return f"[Financial] {symbol}: Giá=${price}, Market Cap={market_cap}"


def rag_agent_stub(question: str) -> str:
    return f"[RAG stub] Tra cứu báo cáo: '{question[:60]}' — thay bằng ChromaDB RAG thật"


def analysis_agent(context: str, question: str) -> str:
    prompt = (
        f"Dựa vào thông tin sau:\n{context}\n\n"
        f"Hãy trả lời ngắn gọn câu hỏi: {question}"
    )
    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return response.text

In [14]:
# ĐÃ CHO SẴN — node wrappers và route_to_agents

def route_to_agents(state: OrchestratorState):
    return state["needed_agents"]


def web_research_node(state: OrchestratorState):
    result = web_research_agent(state["question"])
    print(f"  [web_research_node] chạy")
    return {"web_result": result}


def financial_data_node(state: OrchestratorState):
    result = financial_data_agent(state["symbol"])
    print(f"  [financial_data_node] chạy")
    return {"financial_result": result}


def rag_node(state: OrchestratorState):
    result = rag_agent_stub(state["question"])
    print(f"  [rag_node] chạy")
    return {"rag_result": result}


def analysis_node(state: OrchestratorState):
    parts = []
    if state["web_result"]:      parts.append(state["web_result"])
    if state["financial_result"]: parts.append(state["financial_result"])
    if state["rag_result"]:      parts.append(state["rag_result"])
    context = "\n\n".join(parts)
    result = analysis_agent(context, state["question"])
    return {"final_result": result}

In [15]:
# BƯỚC 1: Viết llm_router_node
#
# Gợi ý:
# - Lấy question từ state
# - Gọi llm_router(question) đã viết ở Phần 2
# - In ra "[Router] Agents cần chạy: ..."
# - Trả về {"needed_agents": agents}

def llm_router_node(state: OrchestratorState):
    question = state["question"]

    agents = llm_router(question)
    print(f"[Router] Agents cần chạy: {agents}")
    return {"needed_agents": agents}

In [16]:
# BƯỚC 2: Build LangGraph
#
# Gợi ý: cấu trúc giống hệt Bài 11, chỉ thay "router" node bằng llm_router_node
#
# graph_builder = StateGraph(OrchestratorState)
#
# Thêm 5 node: llm_router_node, web_research_node, financial_data_node, rag_node, analysis_node
# Edge: START → "router"
# Conditional edge: "router" → route_to_agents
# Edge: mỗi agent node → "analysis"
# Edge: "analysis" → END
#
# graph = graph_builder.compile()

graph_builder = StateGraph(OrchestratorState)

graph_builder.add_node("web_research", web_research_node)
graph_builder.add_node("financial_data", financial_data_node)
graph_builder.add_node("rag", rag_node)
graph_builder.add_node("router", llm_router_node)
graph_builder.add_node("analysis", analysis_node)


graph_builder.add_edge(START, "router")
graph_builder.add_conditional_edges("router", route_to_agents)

graph_builder.add_edge("web_research", "analysis")
graph_builder.add_edge("financial_data", "analysis")
graph_builder.add_edge("rag", "analysis")
graph_builder.add_edge("analysis", END)

graph = graph_builder.compile()

In [ ]:
# BƯỚC 3: Test với các câu hỏi mà keyword router đã fail
# So sánh output của keyword_router và llm_router_node với cùng câu hỏi

def run(question: str, symbol: str = "NVDA"):
    print(f"\n{'='*60}")
    print(f"Câu hỏi: {question}")
    print(f"Keyword router chọn: {keyword_router(question)}")
    initial_state = {
        "question": question,
        "symbol": symbol,
        "needed_agents": [],
        "web_result": "",
        "financial_result": "",
        "rag_result": "",
        "final_result": "",
    }
    result = graph.invoke(initial_state)
    print(f"\nKết quả cuối:\n{result['final_result']}")


# Chạy ít nhất 3 câu — bao gồm câu mà keyword router fail
run("What happened to NVDA last week?")
run("Doanh thu quý 1 2027 của NVIDIA bao nhiêu?")
run("Giá NVDA hôm nay và có tin tức gì không?")


Câu hỏi: What happened to NVDA last week?
Keyword router chọn: ['web_research', 'financial_data', 'rag']
[Router] Agents cần chạy: ['web_research']
  [web_research_node] chạy

Kết quả cuối:
Cổ phiếu NVDA đã tăng giá mạnh vào tuần trước sau báo cáo thu nhập quý 1/2025 vượt xa kỳ vọng của các nhà phân tích, chủ yếu nhờ nhu cầu khổng lồ đối với chip AI của hãng.

Câu hỏi: Doanh thu quý 1 2027 của NVIDIA bao nhiêu?
Keyword router chọn: ['rag']
[Router] Agents cần chạy: ['web_research']
  [web_research_node] chạy

Kết quả cuối:
Doanh thu quý 1 2027 của NVIDIA **chưa được công bố** vì đây là thông tin tài chính của một kỳ trong tương lai. Hiện tại (năm 2024), chúng ta chỉ có thể biết được doanh thu của các quý đã kết thúc và được báo cáo.

Câu hỏi: Giá NVDA hôm nay và có tin tức gì không?
Keyword router chọn: ['web_research', 'financial_data']
[Router] Agents cần chạy: ['financial_data', 'web_research']
  [web_research_node] chạy
  [financial_data_node] chạy

Kết quả cuối:
Giá NVDA hôm nay 